## Read in Data

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

# Provide explicit schema
schema = StructType([
    StructField("Customer_ID",       IntegerType(), False),
    StructField("Age",               IntegerType(), True),
    StructField("Gender",            StringType(),  True),
    StructField("Annual_Income",     IntegerType(),  True),
    StructField("Spending_Score",    IntegerType(), True),
    StructField("Membership_Years",  DoubleType(), True),
    StructField("Online_Purchases",  IntegerType(), True),
    StructField("Discount_Usage",    DoubleType(),  True),
    StructField("Churn",             IntegerType(), True),
])

df = spark.read.csv(
    "/Volumes/workspace/default/ecommerce_data/customer-behavior-and-churn-prediction-dataset/ecommerce_customer_data.csv",
    header="true",
    schema=schema
)

In [0]:
# Set a random seed
RANDOM_SEED = 42

## Train-Test Split

In [0]:
from pyspark.sql import functions as F

def spark_split(df, ratios:list=[0.8, 0.2], target_col:str='Churn', random_seed=RANDOM_SEED):
  pos = df.filter(F.col(target_col)==1)
  neg = df.filter(F.col(target_col)==0)
  
  train_pos, test_pos = pos.randomSplit(ratios, seed=random_seed)
  train_neg, test_neg = neg.randomSplit(ratios, seed=random_seed)
  
  return train_pos.union(train_neg), test_pos.union(test_neg)

In [0]:
train, test = spark_split(df)

train = train.drop('Customer_ID')
test = test.drop('Customer_ID')

# Examine length of train and test sets
print(f"Length of train set: {train.count()}")
print(f"Length of test set: {test.count()}")

## Feature Engineering

#### Transformations and Feature Creation

In [0]:
from pyspark.sql.functions import when

# Convert Gender to Male Binary
train = train.withColumn("Male", when(train.Gender == "Male", 1).otherwise(0))
test = test.withColumn("Male", when(test.Gender == "Male", 1).otherwise(0))

In [0]:
# Polynomial feature for Age
train = train.withColumn("Age_sq", train.Age**2)
test = test.withColumn("Age_sq", test.Age**2)

In [0]:
# Interaction terms for Male with Spending_Score, Discount_Usage, Annual_Income, and Age
train = train.withColumn("Male_Spending_Score", train.Male * train.Spending_Score)
train = train.withColumn("Male_Discount_Usage", train.Male * train.Discount_Usage)
train = train.withColumn("Male_Annual_Income", train.Male * train.Annual_Income)
train = train.withColumn("Male_Age", train.Male * train.Age)

test = test.withColumn("Male_Spending_Score", test.Male * test.Spending_Score)
test = test.withColumn("Male_Discount_Usage", test.Male * test.Discount_Usage)
test = test.withColumn("Male_Annual_Income", test.Male * test.Annual_Income)
test = test.withColumn("Male_Age", test.Male * test.Age)

In [0]:
# Interaction terms derived from EDA
# Interaction term for Membership_Years and Age
train = train.withColumn("Age_Membership_Years", train.Age * train.Membership_Years)
test = test.withColumn("Age_Membership_Years", test.Age * test.Membership_Years)

# Interaction term for Discount_Usage and Online_Purchases
train = train.withColumn("Discount_Usage_Online_Purchases", train.Discount_Usage * train.Online_Purchases)
test = test.withColumn("Discount_Usage_Online_Purchases", test.Discount_Usage * test.Online_Purchases)

# Interaction term for Discount_Usage and Annual_Income
train = train.withColumn("Discount_Usage_Annual_Income", train.Discount_Usage * train.Annual_Income)
test = test.withColumn("Discount_Usage_Annual_Income", test.Discount_Usage * test.Annual_Income)

# Interaction term for Online_Purchases and Annual_Income
train = train.withColumn("Online_Purchases_Annual_Income", train.Online_Purchases * train.Annual_Income)
test = test.withColumn("Online_Purchases_Annual_Income", test.Online_Purchases * test.Annual_Income)

# Interaction term for Membership_Years and Annual_Income
train = train.withColumn("Membership_Years_Annual_Income", train.Membership_Years * train.Annual_Income)
test = test.withColumn("Membership_Years_Annual_Income", test.Membership_Years * test.Annual_Income)

#### Logistic Regression Features

In [0]:
from pyspark.ml.feature import VectorAssembler

# Define the columns to be used as features (logistic regression)
lr_features_all = ["Age_sq", "Age", "Male", "Male_Spending_Score", "Male_Discount_Usage", "Male_Annual_Income", "Male_Age", "Spending_Score", "Discount_Usage", "Annual_Income", "Age_Membership_Years", "Discount_Usage_Online_Purchases", "Discount_Usage_Annual_Income", "Online_Purchases_Annual_Income", "Membership_Years_Annual_Income"]

lr_features = ["Age_sq", "Age", "Annual_Income", "Spending_Score", "Membership_Years", "Online_Purchases", "Discount_Usage"]

# Create a VectorAssembler to combine the features into a single vector column
lr_all_assembler = VectorAssembler(inputCols=lr_features_all, outputCol="features")
lr_assembler = VectorAssembler(inputCols=lr_features, outputCol="features")

# Apply the VectorAssembler to the training and test datasets
lr_all_train = lr_all_assembler.transform(train)
lr_train = lr_assembler.transform(train)

lr_all_test = lr_all_assembler.transform(test)
lr_test = lr_assembler.transform(test)

In [0]:
from pyspark.sql.functions import col

# Create class weight columns
# Count positives and negatives for each train set
lr_all_positives = lr_all_train.filter(col("Churn") == 1).count()
lr_all_negatives = lr_all_train.filter(col("Churn") == 0).count()

lr_positives = lr_train.filter(col("Churn") == 1).count()
lr_negatives = lr_train.filter(col("Churn") == 0).count()

# Create class weights columns for each set (weights sum to 1)
lr_all_total = lr_all_positives + lr_all_negatives
lr_total = lr_positives + lr_negatives

lr_all_train = lr_all_train.withColumn(
    "classWeightCol",
    when(col("Churn")==1, lr_all_negatives / lr_all_total)
    .otherwise(lr_all_positives / lr_all_total)
)
lr_train = lr_train.withColumn(
    "classWeightCol",
    when(col("Churn")==1, lr_negatives / lr_total)
    .otherwise(lr_positives / lr_total)
)

In [0]:
# Select only the columns needed for training and testing
lr_all_train = lr_all_train.select("features", "Churn", "classWeightCol")
lr_train = lr_train.select("features", "Churn", "classWeightCol")

lr_all_test = lr_all_test.select("features", "Churn")
lr_test = lr_test.select("features", "Churn")

#### Other Model Features

In [0]:
# Define the columns to be used as features (other models)
features = ["Age", "Male", "Annual_Income", "Spending_Score", "Membership_Years", "Online_Purchases", "Discount_Usage"]

all_features = ["Age", "Annual_Income", "Spending_Score", "Membership_Years", "Online_Purchases", "Discount_Usage", "Male_Spending_Score", "Male_Discount_Usage", "Male_Annual_Income", "Male_Age", "Age_Membership_Years", "Discount_Usage_Online_Purchases", "Discount_Usage_Annual_Income", "Online_Purchases_Annual_Income", "Membership_Years_Annual_Income"]

# Create a VectorAssembler to combine the features into a single vector column
assembler = VectorAssembler(inputCols=features, outputCol="features")
all_assembler = VectorAssembler(inputCols=all_features, outputCol="features")

# Apply the VectorAssembler to the training and test datasets
train_data = assembler.transform(train)
test_data = assembler.transform(test)

train_data_all = all_assembler.transform(train)
test_data_all = all_assembler.transform(test)

In [0]:
# Create class weights columns
# Count positives and negatives for each train set
train_positives = train_data.filter(col("Churn") == 1).count()
train_negatives = train_data.filter(col("Churn") == 0).count()

train_positives_all = train_data_all.filter(col("Churn") == 1).count()
train_negatives_all = train_data_all.filter(col("Churn") == 0).count()

# Create class weights columns for each set (weights sum to 1)
train_total = train_positives + train_negatives
train_total_all = train_positives_all + train_negatives_all

train_data = train_data.withColumn(
    "classWeightCol",
    when(col("Churn") == 1, train_negatives / train_total)
    .otherwise(train_positives / train_total)
)
train_data_all = train_data_all.withColumn(
    "classWeightCol",
    when(col("Churn") == 1, train_negatives_all / train_total_all)
    .otherwise(train_positives_all / train_total_all)
)

In [0]:
# Select only the columns needed for training and testing
train_data = train_data.select("features", "Churn", "classWeightCol")
test_data = test_data.select("features", "Churn")

train_data_all = train_data_all.select("features", "Churn", "classWeightCol")
test_data_all = test_data_all.select("features", "Churn")